# Endpoint–Artifact Forensic Investigation

This notebook provides the final investigation layer over the Neo4j forensic
graph.

The analysis distinguishes between:

- explicit malicious security evidence
- ordinary endpoint-artifact activity
- activity-based investigation indicators

An explicit malicious detection means that the source endpoint-security
telemetry classified the artifact as `FileType: Malicious`.

Activity concentration or artifact volume is treated only as an
investigation-prioritization indicator. It is not, by itself, proof of
endpoint compromise.

In [ ]:
# Import getpass so the Neo4j password is never stored directly in source code.
import getpass

# Import pandas for converting Cypher results into investigation tables.
import pandas as pd

# Import the Neo4j Python driver for parameterized graph queries.
from neo4j import GraphDatabase


# Centralize the Neo4j connection configuration used by this project.
URI = "neo4j://127.0.0.1:7687"
USERNAME = "neo4j"
DATABASE = "indianarmygraph"


# Prompt securely for the Neo4j password.
PASSWORD = getpass.getpass("Enter the Neo4j password: ")

# Create the driver used by all investigation queries in this notebook.
driver = GraphDatabase.driver(
    URI,
    auth=(USERNAME, PASSWORD),
)

# Verify connectivity before starting investigation queries.
driver.verify_connectivity()

print("Neo4j connection successful.")

## Graph Population

The final graph should contain the complete endpoint population from the
raw security telemetry.

The expected project-level values are:

- 247 endpoints
- 72 endpoint-artifact relationships
- 33 endpoints with explicit malicious detections
- 60 endpoint-artifact relationships carrying explicit malicious evidence

In [ ]:
# Retrieve the primary graph population metrics.
result = driver.execute_query(
    """
    MATCH (e:Endpoint)
    WITH count(e) AS endpoint_count

    MATCH (a:Artifact)
    WITH
        endpoint_count,
        count(a) AS artifact_count

    MATCH ()-[r:OBSERVED]->()
    WITH
        endpoint_count,
        artifact_count,
        count(r) AS relationship_count

    MATCH (e:Endpoint)-[r:OBSERVED]->(a:Artifact)
    WITH
        endpoint_count,
        artifact_count,
        relationship_count,
        count(
            CASE
                WHEN r.malicious_detected = true
                THEN 1
            END
        ) AS malicious_relationships,
        count(
            DISTINCT CASE
                WHEN r.malicious_detected = true
                THEN e
            END
        ) AS malicious_endpoints

    RETURN
        endpoint_count,
        artifact_count,
        relationship_count,
        malicious_relationships,
        malicious_endpoints
    """,
    database_=DATABASE,
)

graph_summary = dict(result.records[0])

print("=" * 70)
print("GRAPH POPULATION")
print("=" * 70)

for key, value in graph_summary.items():
    print(f"{key}: {value}")

In [ ]:
# Retrieve the primary graph population metrics.
result = driver.execute_query(
    """
    MATCH (e:Endpoint)
    WITH count(e) AS endpoint_count

    MATCH (a:Artifact)
    WITH
        endpoint_count,
        count(a) AS artifact_count

    MATCH ()-[r:OBSERVED]->()
    WITH
        endpoint_count,
        artifact_count,
        count(r) AS relationship_count

    MATCH (e:Endpoint)-[r:OBSERVED]->(a:Artifact)
    WITH
        endpoint_count,
        artifact_count,
        relationship_count,
        count(
            CASE
                WHEN r.malicious_detected = true
                THEN 1
            END
        ) AS malicious_relationships,
        count(
            DISTINCT CASE
                WHEN r.malicious_detected = true
                THEN e
            END
        ) AS malicious_endpoints

    RETURN
        endpoint_count,
        artifact_count,
        relationship_count,
        malicious_relationships,
        malicious_endpoints
    """,
    database_=DATABASE,
)

graph_summary = dict(result.records[0])

print("=" * 70)
print("GRAPH POPULATION")
print("=" * 70)

for key, value in graph_summary.items():
    print(f"{key}: {value}")

In [ ]:
# Validate the critical population counts established during graph import.
assert graph_summary["endpoint_count"] == 247
assert graph_summary["relationship_count"] == 72
assert graph_summary["malicious_relationships"] == 60
assert graph_summary["malicious_endpoints"] == 33

print("\nGraph population validation passed.")

## Endpoint Investigation

The following function provides a complete endpoint-level investigation.

The returned summary separates:

- artifact activity
- explicit malicious detections
- observation timing

In [ ]:
def endpoint_summary(endpoint_id):
    """
    Return the main forensic metrics for one endpoint.

    Explicit malicious detections are counted independently from total
    artifact observations so that direct security evidence is not confused
    with activity-based investigation indicators.
    """

    result = driver.execute_query(
        """
        MATCH (e:Endpoint {
            endpoint_id: $endpoint_id
        })

        OPTIONAL MATCH (e)-[r:OBSERVED]->(a:Artifact)

        RETURN
            e.endpoint_id AS endpoint,
            count(DISTINCT a) AS unique_artifacts,
            coalesce(sum(r.record_count), 0) AS total_observations,

            count(
                DISTINCT CASE
                    WHEN r.malicious_detected = true
                    THEN a
                END
            ) AS malicious_artifacts,

            coalesce(
                sum(
                    CASE
                        WHEN r.malicious_detected = true
                        THEN r.malicious_detection_count
                        ELSE 0
                    END
                ),
                0
            ) AS malicious_detections,

            min(r.first_event) AS first_observation,
            max(r.last_event) AS last_observation,

            min(
                CASE
                    WHEN r.malicious_detected = true
                    THEN r.first_malicious_detection
                END
            ) AS first_malicious_detection,

            max(
                CASE
                    WHEN r.malicious_detected = true
                    THEN r.last_malicious_detection
                END
            ) AS last_malicious_detection
        """,
        endpoint_id=endpoint_id,
        database_=DATABASE,
    )

    if not result.records:
        return None

    return dict(result.records[0])

In [ ]:
def investigate_endpoint(endpoint_id):
    """
    Retrieve every artifact observed on the selected endpoint.

    Relationship properties are returned because observation counts,
    malicious classifications, and security timestamps belong to the
    endpoint-artifact observation context.
    """

    result = driver.execute_query(
        """
        MATCH (e:Endpoint {
            endpoint_id: $endpoint_id
        })
        OPTIONAL MATCH (e)-[r:OBSERVED]->(a:Artifact)

        RETURN
            e.endpoint_id AS endpoint,
            a.sha256 AS sha256,
            a.file_names AS file_names,

            r.record_count AS record_count,
            r.first_event AS first_event,
            r.last_event AS last_event,

            coalesce(
                r.malicious_detected,
                false
            ) AS malicious_detected,

            coalesce(
                r.malicious_detection_count,
                0
            ) AS malicious_detection_count,

            r.malicious_file_names AS malicious_file_names,
            r.first_malicious_detection AS first_malicious_detection,
            r.last_malicious_detection AS last_malicious_detection,
            r.first_quarantine_time AS first_quarantine_time,
            r.last_quarantine_time AS last_quarantine_time

        ORDER BY
            r.first_event,
            r.last_event
        """,
        endpoint_id=endpoint_id,
        database_=DATABASE,
    )

    return pd.DataFrame(
        [dict(record) for record in result.records]
    )

In [ ]:
def endpoint_report(endpoint_id):
    """
    Combine the endpoint summary and artifact-level evidence into one report.
    """

    summary = endpoint_summary(endpoint_id)

    if summary is None:
        print(
            f"Endpoint not found: {endpoint_id}"
        )
        return None

    artifacts = investigate_endpoint(endpoint_id)

    return {
        "summary": summary,
        "artifacts": artifacts,
    }

In [ ]:
# Investigate an endpoint known to contain explicit malicious detections.
report = endpoint_report("EPVSWBNA")

print("=" * 70)
print("EPVSWBNA INVESTIGATION")
print("=" * 70)

print("\nSummary:")

for key, value in report["summary"].items():
    print(f"{key}: {value}")

print("\nArtifact evidence:")

display(report["artifacts"])

In [ ]:
# Investigate an endpoint that was confirmed to have no explicit malicious
# security evidence.
report_clean = endpoint_report("EP2C4ZH5")

print("=" * 70)
print("EP2C4ZH5 INVESTIGATION")
print("=" * 70)

for key, value in report_clean["summary"].items():
    print(f"{key}: {value}")

print("\nArtifact evidence:")

display(report_clean["artifacts"])

In [ ]:
# Retrieve every endpoint that has at least one explicit malicious
# security detection stored on an OBSERVED relationship.
result = driver.execute_query(
    """
    MATCH (e:Endpoint)-[r:OBSERVED]->(a:Artifact)

    WHERE r.malicious_detected = true

    RETURN
        e.endpoint_id AS endpoint,
        count(DISTINCT a) AS malicious_artifacts,
        sum(r.malicious_detection_count) AS malicious_detections

    ORDER BY
        malicious_artifacts DESC,
        malicious_detections DESC,
        endpoint
    """,
    database_=DATABASE,
)

malicious_endpoint_df = pd.DataFrame(
    [dict(record) for record in result.records]
)

print("=" * 70)
print("ENDPOINTS WITH EXPLICIT MALICIOUS EVIDENCE")
print("=" * 70)

print(
    "Count:",
    len(malicious_endpoint_df),
)

display(malicious_endpoint_df)

In [ ]:
# Retrieve every endpoint that has at least one explicit malicious
# security detection stored on an OBSERVED relationship.
result = driver.execute_query(
    """
    MATCH (e:Endpoint)-[r:OBSERVED]->(a:Artifact)

    WHERE r.malicious_detected = true

    RETURN
        e.endpoint_id AS endpoint,
        count(DISTINCT a) AS malicious_artifacts,
        sum(r.malicious_detection_count) AS malicious_detections

    ORDER BY
        malicious_artifacts DESC,
        malicious_detections DESC,
        endpoint
    """,
    database_=DATABASE,
)

malicious_endpoint_df = pd.DataFrame(
    [dict(record) for record in result.records]
)

print("=" * 70)
print("ENDPOINTS WITH EXPLICIT MALICIOUS EVIDENCE")
print("=" * 70)

print(
    "Count:",
    len(malicious_endpoint_df),
)

display(malicious_endpoint_df)

In [ ]:
# Rank endpoints by observed artifact activity.
# This is an investigation-prioritization metric and is intentionally
# kept separate from explicit malicious detection evidence.
result = driver.execute_query(
    """
    MATCH (e:Endpoint)-[r:OBSERVED]->(a:Artifact)

    RETURN
        e.endpoint_id AS endpoint,
        count(DISTINCT a) AS unique_artifacts,
        sum(r.record_count) AS total_observations,

        count(
            DISTINCT CASE
                WHEN r.malicious_detected = true
                THEN a
            END
        ) AS malicious_artifacts,

        min(r.first_event) AS first_observation,
        max(r.last_event) AS last_observation

    ORDER BY
        total_observations DESC
    """,
    database_=DATABASE,
)

activity_df = pd.DataFrame(
    [dict(record) for record in result.records]
)

display(activity_df)

In [ ]:
# Investigate the endpoint with the highest observed artifact concentration
# identified during the earlier forensic analysis.
ep4yifhv_report = endpoint_report("EP4YIFHV")

print("=" * 70)
print("EP4YIFHV INVESTIGATION")
print("=" * 70)

for key, value in ep4yifhv_report["summary"].items():
    print(f"{key}: {value}")

print("\nArtifacts:")

display(
    ep4yifhv_report["artifacts"]
)

In [ ]:
# Calculate the duration between an endpoint's first and last observed events.
# Neo4j already stores first_event and last_event as LOCAL DATETIME values,
# so no string conversion is required before calculating the duration.
result = driver.execute_query(
    """
    MATCH (e:Endpoint)-[r:OBSERVED]->(a:Artifact)

    WITH
        e.endpoint_id AS endpoint,
        min(r.first_event) AS first_observation,
        max(r.last_event) AS last_observation

    RETURN
        endpoint,
        first_observation,
        last_observation,

        duration.between(
            first_observation,
            last_observation
        ) AS observation_duration

    ORDER BY observation_duration DESC
    """,
    database_=DATABASE,
)

# Convert the Neo4j records into a pandas DataFrame for inspection.
duration_df = pd.DataFrame(
    [dict(record) for record in result.records]
)

display(duration_df)

In [ ]:
# Identify SHA-256 values observed on more than one endpoint.
# This provides graph-based correlation between endpoint investigations.
result = driver.execute_query(
    """
    MATCH (e:Endpoint)-[:OBSERVED]->(a:Artifact)

    WITH
        a,
        collect(DISTINCT e.endpoint_id) AS endpoints

    WHERE size(endpoints) > 1

    RETURN
        a.sha256 AS sha256,
        a.file_names AS file_names,
        endpoints,
        size(endpoints) AS endpoint_count

    ORDER BY endpoint_count DESC
    """,
    database=DATABASE,
)

shared_artifacts_df = pd.DataFrame(
    [dict(record) for record in result.records]
)

print("=" * 70)
print("CROSS-ENDPOINT ARTIFACT REUSE")
print("=" * 70)

print(
    "Shared artifacts:",
    len(shared_artifacts_df),
)

display(shared_artifacts_df)

In [ ]:
# Validate the final investigation layer against the known project facts.
result = driver.execute_query(
    """
    MATCH (e:Endpoint)-[r:OBSERVED]->(a:Artifact)

    RETURN
        count(DISTINCT e) AS connected_endpoints,
        count(DISTINCT a) AS connected_artifacts,
        count(r) AS observed_relationships,

        count(
            CASE
                WHEN r.malicious_detected = true
                THEN 1
            END
        ) AS malicious_relationships,

        count(
            DISTINCT CASE
                WHEN r.malicious_detected = true
                THEN e
            END
        ) AS malicious_endpoints
    """,
    database_=DATABASE,
)

validation = dict(result.records[0])

print("=" * 70)
print("FINAL INVESTIGATION VALIDATION")
print("=" * 70)

for key, value in validation.items():
    print(f"{key}: {value}")


assert validation["observed_relationships"] == 72
assert validation["malicious_relationships"] == 60
assert validation["malicious_endpoints"] == 33

print("\nInvestigation validation passed.")

In [ ]:
# Close the Neo4j driver after all investigation queries have completed.
driver.close()

print("Neo4j driver closed.")